In [ ]:
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", None)

df_aggregator = pd.read_csv('../data/raw/chainlink_aggregator_raw.csv')
df_aggregator.head()

In [ ]:
chainlink_aggregator_address_mapping = {
    # BTCUSD
    '0xf5fff180082d6017036b771ba883025c654bc935': 'BTCUSD',
    '0x276de5c241071b4728697f4e11a377484a2dd6cb': 'BTCUSD',
    '0xf570deefff684d964dc3e15e1f9414283e3f7419': 'BTCUSD',
    '0x7104ac4abcecf1680f933b04c214b0c491d43ecc': 'BTCUSD',
    '0xae74faa92cb67a95ebcab07358bc222e33a34da7': 'BTCUSD',
    '0xdbe1941bfbe4410d6865b9b7078e0b49af144d2d': 'BTCUSD',
    '0x4a3411ac2948b33c69666b35cc6d055b27ea84f1': 'BTCUSD',
    # ETHUSD
    '0xf79d6afbb6da890132f9d7c355e3015f15f3406f': 'ETHUSD',
    '0xb103ede8acd6f0c106b7a5772e9d24e34f5ebc2c': 'ETHUSD',
    '0x00c7a37b03690fb9f41b5c5af8131735c7275446': 'ETHUSD',
    '0xd3fcd40153e56110e6eeae13e12530e26c9cb4fd': 'ETHUSD',
    '0x37bc7498f4ff12c19678ee8fe19d713b87f6a9e6': 'ETHUSD',
    '0xe62b71cf983019bff55bc83b48601ce8419650cc': 'ETHUSD',
    # LINKUSD
    '0x32dbd3214ac75223e27e575c53944307914f7a90': 'LINKUSD',
    '0xc8b5381a98c7dc8b91f4149303397da56061ebaf': 'LINKUSD',
    '0x8cde021f0bfa5f82610e8ce46493cf66ac04af53': 'LINKUSD',
    '0xbd11bc57fc140614190cabc1b4c316aba220bae4': 'LINKUSD',
    '0xdfd03bfc3465107ce570a0397b247f546a42d0fa': 'LINKUSD',
    '0x20807cf61ad17c31837776fa39847a2fa1839e81': 'LINKUSD',
    '0x96d6e33b411dc1f4e3f1e894a5a5d9ce0f96738d': 'LINKUSD',
    # USDCUSD
    '0x3b15a92872435c01c27201aae0968839fb45217d': 'USDCUSD',
    '0x789190466e21a8b78b8027866cbbdc151542a26c': 'USDCUSD',
    '0xc9e1a09622afdb659913fefe800feae5dbbfe9d7': 'USDCUSD',
}

In [ ]:
from helper import hex_to_uint, hex_to_int_signed_256, first_data_word, split_abi_words

In [ ]:
df_aggregator['block_timestamp'] = pd.to_datetime(df_aggregator["block_timestamp"], errors="coerce", utc=True)
df_aggregator["date"] = df_aggregator["block_timestamp"].dt.date
df_aggregator["minute"] = df_aggregator["block_timestamp"].dt.floor("min")

df_aggregator['record_type'] = 'chainlink_oracle_update'
df_aggregator['feed_name'] = df_aggregator['address'].map(chainlink_aggregator_address_mapping)

# Decode Chainlink fields
df_aggregator["oracle_price_raw"] = df_aggregator["topic1"].apply(hex_to_int_signed_256)
df_aggregator["oracle_price"] = df_aggregator["oracle_price_raw"] / 1e8

df_aggregator["round_id"] = df_aggregator["topic2"].apply(hex_to_uint)

df_aggregator["updated_at_raw_hex"] = df_aggregator["data"].apply(first_data_word)
df_aggregator["updated_at_unix"] = df_aggregator["updated_at_raw_hex"].apply(hex_to_uint)
df_aggregator["updated_at_datetime"] = pd.to_datetime(
    df_aggregator["updated_at_unix"],
    unit="s",
    errors="coerce",
    utc=True
)

In [ ]:
word_lists = df_aggregator["data"].apply(split_abi_words)

max_words = word_lists.apply(len).max()

# Create decoded columns
for i in range(int(max_words)):
    df_aggregator[f"data_word_{i}"] = word_lists.apply(lambda w: w[i] if len(w) > i else np.nan)
    df_aggregator[f"data_word_{i}_uint"] = df_aggregator[f"data_word_{i}"].apply(hex_to_uint)
    df_aggregator[f"data_word_{i}_int"] = df_aggregator[f"data_word_{i}"].apply(hex_to_int_signed_256)

In [ ]:
df_aggregator.head()

In [ ]:
df_aggregator.to_csv('../data/processed/chainlink_aggregator_processed.csv', index=False)

In [ ]:
df_binance = pd.read_csv('../data/processed/binance_benchmark_1m.csv')
df_binance['minute'] = pd.to_datetime(df_binance['minute'], errors='coerce', utc=True)

feed_to_symbol = {
    'BTCUSD': 'BTCUSDT',
    'ETHUSD': 'ETHUSDT',
    'LINKUSD': 'LINKUSDT',
    'USDCUSD': 'USDCUSDT'
}

df_aggregator['symbol'] = df_aggregator['feed_name'].map(feed_to_symbol)

df_oracle_merged = df_aggregator.dropna(subset=['symbol']).merge(
    df_binance[['symbol', 'minute', 'benchmark_close']],
    on=['symbol', 'minute'],
    how='left'
)

In [ ]:
df_oracle_merged = df_oracle_merged.dropna(subset=['oracle_price', 'benchmark_close']).copy()

df_oracle_merged['oracle_price'] = pd.to_numeric(df_oracle_merged['oracle_price'], errors='coerce')
df_oracle_merged['benchmark_close'] = pd.to_numeric(df_oracle_merged['benchmark_close'], errors='coerce')

df_oracle_merged = df_oracle_merged[
    (df_oracle_merged['oracle_price'] > 0) &
    (df_oracle_merged['benchmark_close'] > 0)
].copy()

# Oracle deviation = log(oracle_price) - log(benchmark_close)
df_oracle_merged['oracle_error'] = (
    np.log(df_oracle_merged['oracle_price']) -
    np.log(df_oracle_merged['benchmark_close'])
)

df_oracle_merged['abs_oracle_error'] = df_oracle_merged['oracle_error'].abs()
df_oracle_merged['oracle_error_pct'] = df_oracle_merged['oracle_error'] * 100
df_oracle_merged['abs_oracle_error_pct'] = df_oracle_merged['abs_oracle_error'] * 100

# Shock flags
df_oracle_merged['shock_05pct'] = (df_oracle_merged['abs_oracle_error'] > 0.005).astype(int)
df_oracle_merged['shock_1pct'] = (df_oracle_merged['abs_oracle_error'] > 0.010).astype(int)
df_oracle_merged['shock_2pct'] = (df_oracle_merged['abs_oracle_error'] > 0.020).astype(int)
df_oracle_merged['shock_5pct'] = (df_oracle_merged['abs_oracle_error'] > 0.050).astype(int)

In [ ]:
df_oracle_merged.head()

In [ ]:
df_oracle_merged.to_csv('../data/processed/oracle_deviations.csv', index=False)